In [1]:
import jax
import netket as nk

import numpy as np
import jax.numpy as jnp

# from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule
from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule 

In [2]:
mol, mo_coeff, mf = PCMolecule.molecule(cid=947)#62714
molecule = PCMolecule(mol=mol, mo_coeff=mo_coeff)

H = molecule.hamiltonian.to_jax_operator()
hi = molecule.hilbert_space

Hartree-Fock energy: -107.49896754458388
E(CCSD) = -107.6560799974465  E_corr = -0.1571124528626242
CCSD energy: -107.6560799974465


/Users/lucagravina/venvs/neuralimportancesampling/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [3]:
all_states = hi.all_states()
print("all_states_size =", all_states.shape)

print("n_fermions =", hi.n_fermions)
print("n_orbitals =", hi.n_orbitals)

n_max = hi.n_orbitals * 2

all_states_size = (14400, 20)
n_fermions = 14
n_orbitals = 10


In [4]:
_operator_data = H._operator_data
print("_operator_data keys =", _operator_data.keys())

_operator_data keys = dict_keys(['diag', 'offdiag'])


In [5]:
_operator_data['offdiag'].keys()

dict_keys([2, 4])

In [6]:
from jax import Array
from functools import partial
@jax.jit
def hamming_distance(x, y):
    """
    Args:
        x, y: 1D JAX arrays of 0/1 values with the same shape.

    Returns:
      d : Hamming distance (scalar)
    """
    return jnp.sum(x != y)
  
@partial(jax.jit, static_argnames=['k'])
def select_changes(x: Array, y: Array, k: int = 4):
    diff = x - y  
    size = k // 2
    k_destroy = jnp.argwhere(diff == 1, size=size, fill_value=-1).reshape(-1)
    l_create = jnp.argwhere(diff == -1, size=size, fill_value=-1).reshape(-1)
    return jnp.flip(k_destroy), jnp.flip(l_create)

In [16]:
x = all_states[10]
xp, mels = H.get_conn_padded(x)

print(x,"\n")

y = jnp.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1], dtype=jnp.int8)
print(y,"\n")

is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))[0]
    y_index = y_index[0]
    print("matrix element connecting x to y =", mels[y_index])


print("\nHamming distance =", hamming_distance(x, y))

[0 0 0 1 1 1 1 1 1 1 0 1 0 1 1 0 1 1 1 1] 

[0 0 0 1 1 1 1 1 1 1 0 1 0 1 0 1 1 1 1 1] 

Is y in xp? True
matrix element connecting x to y = -2.0816681711721685e-17

Hamming distance = 2


In [8]:
k_destroy, l_create = select_changes(x, y)
print("x = ", x)
print("y = ", y)
print("Deletions indices =", k_destroy)
print("Creations indices =", l_create)

x =  [0 0 0 1 1 1 1 1 1 1 0 1 0 1 1 0 1 1 1 1]
y =  [1 0 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 0 0 0]
Deletions indices = [8 7]
Creations indices = [2 0]


In [10]:
index_array, create_array, weight_array = _operator_data['offdiag'][4]
index_array.shape

(20, 20)

In [11]:
ind = index_array[tuple(k_destroy)]
weight_array[ind]

Array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0.], dtype=float64)

In [15]:
creates = create_array[ind]  # shape (n_max, 2)
creates

Array([[10,  0],
       [10,  2],
       [10,  6],
       [11,  1],
       [11,  3],
       [11,  9],
       [12,  0],
       [12,  2],
       [12,  6],
       [13,  1],
       [13,  3],
       [13,  9],
       [15,  5],
       [16,  0],
       [16,  2],
       [16,  6],
       [17,  7],
       [17,  8],
       [18,  7],
       [18,  8],
       [19,  1],
       [19,  3],
       [19,  9],
       [ 0,  0],
       [ 0,  0]], dtype=int64)

In [16]:
base = n_max # or just use 12
creates_1d = creates[:, 0] * base + creates[:, 1]
target_1d = l_create[0] * base + l_create[1]  # 9*12 + 3 = 111

idx = jnp.searchsorted(creates_1d, target_1d)

In [17]:
print("weight_array shape =", weight_array.shape)
weight_array[ind,idx]

weight_array shape = (191, 25)


Array(-0.03325862, dtype=float64)

In [18]:
@jax.jit
def jw_sign_fast(x, k_destroy, l_create):
    """Vectorized JW sign computation."""
    # Cumsum gives number of particles to the left of each site
    cumsum = jnp.cumsum(x)
    
    # Parity from destruction (use cumsum - x to exclude site itself)
    left_counts = jnp.concatenate([jnp.array([0]), cumsum[:-1]])
    parity_destroy = jnp.sum(left_counts[k_destroy])
    
    # After destruction
    xd = x.at[k_destroy].set(0)
    cumsum_d = jnp.cumsum(xd)
    left_counts_d = jnp.concatenate([jnp.array([0]), cumsum_d[:-1]])
    parity_create = jnp.sum(left_counts_d[l_create])
    
    return 1 - 2 * ((parity_destroy + parity_create) % 2)


jw_sign_fast(x, k_destroy, l_create)

Array(-1, dtype=int64)

In [66]:
x = all_states[10]
xp, mels = H.get_conn_padded(x)

print(x,"\n")

y = jnp.array([0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1], dtype=jnp.int8)
print(y,"\n")

is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))[0]
    y_index = y_index[0]
    print("matrix element connecting x to y =", mels[y_index])


print("\nHamming distance =", hamming_distance(x, y))

[0 0 0 1 1 1 1 1 1 1 0 1 0 1 1 0 1 1 1 1] 

[0 0 0 1 1 1 1 1 1 1 0 1 0 1 0 1 1 1 1 1] 

Is y in xp? True
matrix element connecting x to y = -2.0816681711721685e-17

Hamming distance = 2


In [59]:
k_destroy_, l_create_ = select_changes(x, y, k=2) # identifies the two sites that are different
print("k_destroy_ =", k_destroy_)
print("l_create_ =", l_create_)

k_destroy_ = [14]
l_create_ = [15]


In [60]:
same_sites = jnp.where((x & y), size=hi.n_fermions-1, fill_value=-1)[0] # all sites where a particle could have been both destroyed and created
same_sites

Array([ 3,  4,  5,  6,  7,  8,  9, 11, 13, 16, 17, 18, 19], dtype=int64)

In [70]:
mel = 0.0
for i in same_sites:
    k_destroy = jnp.sort(jnp.array([k_destroy_[0], i]), descending=True)
    l_create = jnp.sort(jnp.array([l_create_[0], i]), descending=True)
    ind = index_array[tuple(k_destroy)]
    
    creates = create_array[ind] # shape (n_max, 4)
        
    mask = jnp.all(creates == l_create[..., None, :], axis=-1)
    idx = jnp.argmax(mask)
    found = mask[idx]
    
    sgn = jw_sign_fast(x, k_destroy, l_create)
    mel_i = jnp.where(found, sgn * weight_array[ind, idx], 0.0)
    print(f"i={i}, k_destroy={k_destroy}, l_create={l_create}, mel_i={mel_i}, sgn={sgn}, ind={ind}, idx={idx}")
    mel += mel_i

i=3, k_destroy=[14  3], l_create=[15  3], mel_i=0.0, sgn=-1, ind=95, idx=0
i=4, k_destroy=[14  4], l_create=[15  4], mel_i=0.0, sgn=-1, ind=96, idx=0
i=5, k_destroy=[14  5], l_create=[15  5], mel_i=0.0, sgn=-1, ind=97, idx=0
i=6, k_destroy=[14  6], l_create=[15  6], mel_i=0.0, sgn=-1, ind=98, idx=0
i=7, k_destroy=[14  7], l_create=[15  7], mel_i=-0.008942072215124004, sgn=-1, ind=99, idx=13
i=8, k_destroy=[14  8], l_create=[15  8], mel_i=0.008942072215124029, sgn=-1, ind=100, idx=14
i=9, k_destroy=[14  9], l_create=[15  9], mel_i=0.0, sgn=-1, ind=101, idx=0
i=11, k_destroy=[14 11], l_create=[15 11], mel_i=0.0, sgn=-1, ind=103, idx=0
i=13, k_destroy=[14 13], l_create=[15 13], mel_i=0.0, sgn=-1, ind=105, idx=0
i=16, k_destroy=[16 14], l_create=[16 15], mel_i=0.0, sgn=-1, ind=135, idx=0
i=17, k_destroy=[17 14], l_create=[17 15], mel_i=0.024316547043329483, sgn=-1, ind=151, idx=6
i=18, k_destroy=[18 14], l_create=[18 15], mel_i=-0.02431654704332953, sgn=-1, ind=168, idx=8
i=19, k_destroy=[

In [71]:
mel

Array(-2.08166817e-17, dtype=float64)

In [ ]:
import jax
import jax.numpy as jnp
from functools import partial

from netket.jax import COOArray
from netket.utils.types import Array

@jax.jit
def hamming_distance(x, y):
    """
    Args:
        x, y: 1D JAX arrays of 0/1 values with the same shape.

    Returns:
      d : Hamming distance (scalar)
    """
    return jnp.sum(x != y)
  
  
@partial(jax.jit, static_argnames=['k'])
def select_changes(x: Array, y: Array, k: int = 4):
    diff = x - y  
    size = k // 2
    k_destroy = jnp.argwhere(diff == 1, size=size, fill_value=-1).reshape(-1)
    l_create = jnp.argwhere(diff == -1, size=size, fill_value=-1).reshape(-1)
    return jnp.flip(k_destroy), jnp.flip(l_create)


@jax.jit
def jw_sign_fast(x, k_destroy, l_create):
    """
    Fast and correct Jordan–Wigner sign that matches the original implementation.
    """

    prefix = jnp.cumsum(x) - x # number of ones to the left
    parity_destroy = jnp.sum(prefix[k_destroy])

    xd = x.at[k_destroy].set(0) # apply destruction BEFORE computing create parity
    prefix_d = jnp.cumsum(xd) - xd # prefix after destruction

    parity_create = jnp.sum(prefix_d[l_create])

    total_parity = (parity_destroy + parity_create) & 1 # total parity
    return 1 - 2 * total_parity # (-1)^parity


@partial(jax.jit, static_argnames=['n_fermions'])
@partial(jnp.vectorize, signature="(n)->()", excluded=(0, 1, 3, 4, 5))
def _get_mel_offdiag(
    n_fermions: int,
    x: Array,
    y: Array,
    index_array: Array | COOArray | None,
    create_array: Array | None,
    weight_array: Array,
):
    r"""
    Get the matrix element between two states `x` and `y` for two-body operators
    of the form 
    
    .. math::
        c^\dagger_{i} c^\dagger_{j} c_{k} c_{l}
        
    Args:
        x: Array
            Initial state (1D array of 0/1 values).
        y: Array
            Final state (1D array of 0/1 values).
        index_array: Array | COOArray | None
            Precomputed index array for the two-body operator.
        create_array: Array | None
            Precomputed creation array for the two-body operator.
        weight_array: Array
            Precomputed weight array for the two-body operator.
            
    Returns:
        mel: Array
            The matrix element connecting `x` to `y`.
    """
    def compute(k_destroy, l_create, ind):
        creates = create_array[ind] # shape (n_max, n_fermions)
        
        mask = jnp.all(creates == l_create[..., None, :], axis=-1)
        idx = jnp.argmax(mask)
        found = mask[idx]
        
        sgn = jw_sign_fast(x, k_destroy, l_create)
        return jnp.where(found, sgn * weight_array[ind, idx], 0.0)

    def case_k4():
        r"""
        Handles the case where the Hamming distance between `x` and `y` is 4,
        corresponding to two-body operator transitions between different sites, i.e
        
        .. math::
            c^\dagger_{i} c^\dagger_{j} c_{k} c_{l} with i,j,k,l all different.
            
        An example of such a transition is:
        x = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
        y = [1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0]
        """
        k_destroy, l_create = select_changes(x, y, k=4)
        ind = index_array[tuple(k_destroy)]
        return compute(k_destroy, l_create, ind)

    def case_k2():
        r"""
        Handles the case where the Hamming distance between `x` and `y` is 2,
        corresponding to two-body operator transitions that involve the destruction
        and creation of a particle in the same site, i.e
        
        .. math::
            c^\dagger_{i} c^\dagger_{j} c_{j} c_{k}  or  c^\dagger_{i} c^\dagger_{i} c_{k} c_{l}
            
        An example of such a transition is:
        x = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
        y = [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]
        """
        k_destroy_, l_create_ = select_changes(x, y, k=2) # identifies the two sites that are different
        same_sites = jnp.where((x & y), size=n_fermions-1, fill_value=-1)[0] # all sites where a particle could have been both destroyed and created
        
        def f_one_site(i):
            r"""
            Computes the matrix element contribution for a single same-site transition.
            Args:
                i: Index of the site where a particle is both destroyed and created.
            Returns:
                Matrix element contribution for this specific same-site transition.
            """
            k_destroy = jnp.sort(jnp.array([k_destroy_[0], i]), descending=True)
            l_create = jnp.sort(jnp.array([l_create_[0], i]), descending=True)
            ind = index_array[tuple(k_destroy)]
            return compute(k_destroy, l_create, ind)

        mels = jax.vmap(f_one_site)(same_sites) # vectorize over all i in same_sites
        valid_mask = same_sites != -1 # mask out padding (-1 entries)
        return jnp.sum(mels * valid_mask)

    d = hamming_distance(x, y)
    return jnp.where(d == 4, case_k4(), jnp.where(d == 2, case_k2(), 0.0))



In [73]:
import jax.ops
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc

def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            k: v for k, v in inner_dict.items() 
            if filter_func(k)
        }
    return result


_operator_data_filtered = filter_keys(_operator_data, lambda k: k == 4)
_operator_data_filtered['diag'] = {}

x = all_states[10]

xp, mels = get_conn_padded_pnc(_operator_data_filtered, x, hi.n_fermions)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

print("xp shape =", xp.shape)
print("mels shape =", mels.shape)

xp shape = (127, 20)
mels shape = (127,)


In [74]:
k = 4
index_array, create_array, weight_array = _operator_data_filtered['offdiag'][4]

mels_off_diag = _get_mel_offdiag(
    hi.n_fermions,
    x,
    xp,
    index_array,
    create_array,
    weight_array,
)

jnp.where(mels - mels_off_diag != 0)

(Array([], shape=(0,), dtype=int64),)